# SynthFace on Google Colab: approach B with many identities (EXPERIMENT ONLY)

For each synthetic identity: Blender renders 4 fixed shots (frontal reference + yaw +15 / yaw -15 /
pitch +10 warm light) with depth and hair masks, then Stable Diffusion 1.5 + ControlNet-depth makes
the frontal shot photoreal (the identity reference) and IP-Adapter Plus-Face carries that identity
into the other shots. The report checks whether each person stays the same person.

**Rules (CLAUDE.md rule 1):** AI images are experiment-only. They stay in `output/ai_experiment/` (here: on
your Drive) and must **not** go into training data or `output/<job_id>/`. No real photos as input.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU` (or better).
2. Nothing to set up for the code: it is cloned from the public GitHub repo (no token).
3. Run the cells top to bottom. Re-running a cell resumes (finished shots/images are skipped).

## 1. Settings

In [ ]:
SET = "colab_b_10"              # name of this experiment set
SEED = 20261013                 # identities = sample_identity(SEED, 1..N), same as approach A
PEOPLE = "1-10"                 # identity indexes, e.g. "1-10" or "1,4,7"
SHOTS = "s1,s2,s3,s6"           # s1 must be included (reference)
RENDER_DEVICE = "OPTIX"         # Blender: "OPTIX" or "CUDA"
SD_STEPS = 30                   # DPM++ steps x strength 0.55 (laptop CPU used 18)
BLIND_PAIRS = 12                # blind test: pairs per kind (same / different)
REPO = "https://github.com/ankitscode/-synthetic-Image-Genrator.git"   # public, no token needed
BRANCH = "experiment/approach-b-img2img"
DRIVE_DIR = "/content/drive/MyDrive/synthface_output"   # None = keep output only in this VM

PROJECT = "/content/synthface"
TOOLS = "/content/synthface_tools"
BLENDER = f"{TOOLS}/blender-5.2.2-linux-x64/blender"
SET_DIR = f"{PROJECT}/output/ai_experiment/sets/{SET}"

## 2. Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 3. Mount Drive and clone the experiment branch (public GitHub repo)

In [ ]:
import os, subprocess

if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", PROJECT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


## 4. Install Blender + MPFB + assets (3-5 min) and the pinned AI packages

In [ ]:
!bash {PROJECT}/colab/setup_colab.sh {TOOLS}
!pip -q install -r {PROJECT}/tools/ai_experiment/requirements_ai.txt

## 5. Send output to Drive (optional)

In [ ]:
out = f"{PROJECT}/output"
if DRIVE_DIR and not os.path.islink(out):
    if os.path.isdir(out) and os.listdir(out):
        raise SystemExit(f"{out} already has files; move them first")
    if os.path.isdir(out):
        os.rmdir(out)
    os.symlink(DRIVE_DIR, out)
os.makedirs(out, exist_ok=True)
print(out, "->", os.path.realpath(out))

## 6. Blender: 3D shots + depth / face / hair masks

One line per finished person. Rendering is on the GPU; the depth and mask maps are ray-cast on the
CPU (~30-60 s per shot on Colab's 2 vCPUs). Log: `<SET_DIR>/render_shots.log`.

In [ ]:
import time
os.makedirs(SET_DIR, exist_ok=True)
cmd = [BLENDER, "--background", "--python", "tools/ai_experiment/render_shots.py", "--",
       "--set", SET, "--seed", str(SEED), "--people", PEOPLE, "--shots", SHOTS, "--device", RENDER_DEVICE]
t0 = time.time()
with open(f"{SET_DIR}/render_shots.log", "a") as log:
    proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=log, text=True)
    for line in proc.stdout:
        if line.startswith(("PERSON_DONE", "Traceback", "Error")):
            print(f"{line.strip()}  {time.time() - t0:.0f}s", flush=True)
print("Blender exit code:", proc.wait())

## 7. Stable Diffusion: references, variations, report

First run downloads SD 1.5 + ControlNet-depth + IP-Adapter at pinned revisions (~6 GB, a few minutes).
Then about 5-10 s per image on a T4.

In [ ]:
!cd {PROJECT} && python tools/ai_experiment/blind_test.py --set {SET} --steps {SD_STEPS} --pairs {BLIND_PAIRS} 2>&1 | grep -vE "it/s|Loading|Fetching"

## 8. Look at the results

In [ ]:
import json
from IPython.display import Image, display
summary = json.load(open(f"{SET_DIR}/b/metrics.json"))["summary"]
print(json.dumps(summary, indent=2))
display(Image(f"{SET_DIR}/b/overview.png"))

## 9. Bring the set back to the laptop

With Drive it is already in `MyDrive/synthface_output/ai_experiment/sets/<SET>/`. Otherwise download the zip below.
On the laptop put it at `output/ai_experiment/sets/<SET>/` (git-ignored) and re-run the report there:
`venv_ai/bin/python tools/ai_experiment/blind_test.py --set <SET> --report`.
Answer `b/blind_B.png` and `b/blind_A.png` before opening `b/blind_answers.json`.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{SET}", "zip", SET_DIR)
files.download(zip_path)